## Load User Data

In [11]:
BANNED_USERS = {51, 86, 62, 39, 66, 89, 67, 57} # failed attention checks

import json
with open('../data/user_code.json') as json_data:
    d = json.load(json_data)
    user_to_scores = {
        row['user_id']: {
            'background': float(row['background_ability']),
            'comprehension': float(row['authorship_and_comprehension'])
        } for row in d if row['user_id'] not in BANNED_USERS
    }
    user_id_to_group = {
        row['user_id']: row['experiment_group'] for row in d if row['user_id'] not in BANNED_USERS
    }

## Prompt Statistics

In [12]:
import json
import pandas as pd

user_to_num_prompts = dict()
with open('../data/ai_trace.json') as json_data:
    data = json.load(json_data)
    for row in data:
        if row['user_id'] in BANNED_USERS:
            continue
        key = (row['user_id'], row['project_id'])
        user_to_num_prompts[key] = user_to_num_prompts.get(key, 0) + 1
        
prompt_dataset = {
    'user': [],
    'key': [],
    'num': [],
}
for (user, task), v in user_to_num_prompts.items():
    prompt_dataset['user'].append(user)
    prompt_dataset['key'].append((user_id_to_group[user], task))
    prompt_dataset['num'].append(v)
prompt_dataset = pd.DataFrame(prompt_dataset)

from scipy.stats import sem
print("Prompt Usage:")
prompt_dataset.groupby("key")["num"].agg(mean="mean", stderr=sem).reset_index()

Prompt Usage:


,key,mean,stderr
0,"(agent, extension)",6.782609,0.882318
1,"(agent, initial)",5.769231,0.441353
2,"(chat, extension)",4.736842,0.576550
3,"(chat, initial)",11.571429,2.505776


## Analyze Agent Prompts

In [13]:
import json

with open('../llm_judge/results/cluster/label_agent_prompts.jsonl', 'r') as json_file:
    json_list = list(json_file)

label_to_users = dict()
    
for json_str in json_list:
    result = json.loads(json_str)
    if result['user_id'] in BANNED_USERS:
        continue
    label = result['label']
    arr = label_to_users.get(label, [])
    arr.append(result['user_id'])
    label_to_users[label] = arr
    
# we merged the 'copy' and 'rephrased' labels, due to low human agreement and realizing that they test the same prompt interaction type
label_to_users['copy'] = label_to_users['copy'] + label_to_users['rephrased']
del label_to_users['rephrased']
label_to_users_set = {k: set(v) for k, v in label_to_users.items()}
total_num = sum([len(v) for v in label_to_users.values()])

In [14]:
import numpy as np
print("## Table 1\n======================")
for label, users in label_to_users_set.items():
    user_bg = [user_to_scores[user]['background'] for user in users]
    user_comp = [user_to_scores[user]['comprehension'] for user in users]
    
    print("Label:", label)
    print("Comp:", np.mean(user_comp))
    print("BG:", np.mean(user_bg))
    print("# Users:", len(users))
    print("Proportion:", (1.0 * len(label_to_users[label])) / total_num)
    print('----------------------')

## Table 1
Label: technical
Comp: 0.7428451178451181
BG: 0.6837606837606837
# Users: 9
Proportion: 0.08666666666666667
----------------------
Label: debugging
Comp: 0.6916666666666668
BG: 0.6615384615384615
# Users: 5
Proportion: 0.04
----------------------
Label: copy
Comp: 0.6540909090909092
BG: 0.6676923076923077
# Users: 25
Proportion: 0.7933333333333333
----------------------
Label: exploratory
Comp: 0.6351010101010102
BG: 0.6923076923076922
# Users: 6
Proportion: 0.04666666666666667
----------------------
Label: other
Comp: 0.7666666666666668
BG: 0.8
# Users: 5
Proportion: 0.03333333333333333
----------------------


In [ ]:
# Regression to confirm

## Analyze Chatbot Prompts

In [5]:
import json

with open('../llm_judge/results/cluster/label_chatbot_prompts.jsonl', 'r') as json_file:
    json_list = list(json_file)

label_to_users = dict()
    
for json_str in json_list:
    result = json.loads(json_str)
    if result['user_id'] in BANNED_USERS:
        continue
    label = result['label']
    arr = label_to_users.get(label, [])
    arr.append(result['user_id'])
    label_to_users[label] = arr
    
label_to_users_set = {k: set(v) for k, v in label_to_users.items()}
total_num = sum([len(v) for v in label_to_users.values()])

In [6]:
import numpy as np
print("## Appendix Table 4\n======================")
for label, users in label_to_users_set.items():
    user_bg = [user_to_scores[user]['background'] for user in users]
    user_comp = [user_to_scores[user]['comprehension'] for user in users]
    
    print("Label:", label)
    print("Comp:", np.mean(user_comp))
    print("BG:", np.mean(user_bg))
    print("# Users:", len(users))
    print("Proportion:", (1.0 * len(label_to_users[label])) / total_num)
    print('----------------------')

## Appendix Table 4
Label: syntax_help
Comp: 0.8255561568061568
BG: 0.6446886446886448
# Users: 21
Proportion: 0.4279835390946502
----------------------
Label: design_help
Comp: 0.8335182178932179
BG: 0.6263736263736265
# Users: 14
Proportion: 0.1934156378600823
----------------------
Label: clarification
Comp: 0.8112201561065198
BG: 0.5804195804195804
# Users: 11
Proportion: 0.16872427983539096
----------------------
Label: snippet
Comp: 0.8527076318742987
BG: 0.658119658119658
# Users: 9
Proportion: 0.09465020576131687
----------------------
Label: debugging
Comp: 0.7805555555555554
BG: 0.49230769230769234
# Users: 5
Proportion: 0.05761316872427984
----------------------
Label: jailbreaking
Comp: 0.7545454545454546
BG: 0.5692307692307692
# Users: 5
Proportion: 0.037037037037037035
----------------------
Label: urgency
Comp: 0.583333333333333
BG: 0.46153846153846156
# Users: 1
Proportion: 0.0205761316872428
----------------------


## AI Review Types

In [7]:
import json
with open('../data/ai_approval.json') as json_data:
    approval_data = json.load(json_data)

In [8]:
# build a user history of actions
user_history = dict()
approval_data.sort(key=lambda item: item['created_at'])
for row in approval_data:
    user_id = row['user_id']
    if user_id in BANNED_USERS:
        continue
    curr_history = user_history.get(user_id, [])
    curr_history.append(row['mode'])
    user_history[user_id] = curr_history

In [9]:
def classify_span(curr_span):
    label = ''
    if 'diff' in curr_span or 'reject' in curr_span or 'reject_all' in curr_span:
        label = 'manual_edit'
    elif 'keep_all' in curr_span:
        label = 'click_accept_all'
    elif 'keep' in curr_span or 'keep_all' in curr_span:
        label = 'click_accept_each'
    else:
        label = 'auto_accept'
    return label

review_label_to_users = {
    'auto_accept': [],
    'manual_edit': [],
    'click_accept_each': [],
    'click_accept_all': [],
}

for user, hist in user_history.items():
    ai_idxs = [idx for idx, elem in enumerate(hist) if elem == 'AI']
    if not ai_idxs:
        continue
        
    start_idx = ai_idxs.pop(0)
    while ai_idxs:
        end_idx = ai_idxs.pop(0)
        span = hist[start_idx:end_idx]
        review_label_to_users[classify_span(span)].append(user)
        start_idx = end_idx
        
    span = hist[start_idx:]
    review_label_to_users[classify_span(span)].append(user)
    
review_label_to_users_set = {k: set(v) for k, v in review_label_to_users.items()}
total_num = sum([len(v) for v in review_label_to_users.values()])

In [10]:
import numpy as np
print("## Table 2\n======================")
for label, users in review_label_to_users_set.items():
    user_bg = [user_to_scores[user]['background'] for user in users]
    user_comp = [user_to_scores[user]['comprehension'] for user in users]
    
    print("Label:", label)
    print("Comp:", np.mean(user_comp))
    print("BG:", np.mean(user_bg))
    print("# Users:", len(users))
    print("Proportion:", (1.0 * len(review_label_to_users[label])) / total_num)
    print('----------------------')

## Table 2
Label: auto_accept
Comp: 0.6145833333333335
BG: 0.6025641025641025
# Users: 6
Proportion: 0.05303030303030303
----------------------
Label: manual_edit
Comp: 0.6609848484848484
BG: 0.6820512820512821
# Users: 15
Proportion: 0.3181818181818182
----------------------
Label: click_accept_each
Comp: 0.7765151515151516
BG: 0.7142857142857143
# Users: 7
Proportion: 0.13636363636363635
----------------------
Label: click_accept_all
Comp: 0.6644886363636363
BG: 0.6692307692307693
# Users: 20
Proportion: 0.49242424242424243
----------------------
